# Polarization analysis

Do bots change the community structure of the reply network? We detect communities with Louvain on the
full reply graph, remove the accounts BotMoE labelled as bots, and run it again (paper, Sections 4.5 and 5.2).

This notebook runs on the **sample** in `preprocessing/sample-data/` with NetworkX + python-louvain.
The numbers in the paper come from the full 4.3M-user graph, which we processed with cuGraph.

In [ ]:
import networkx as nx
import pandas as pd
import community as community_louvain  # python-louvain

PREDICTIONS = "../BotMoE/artifacts/predictions/test_predictions.csv"
NODES = "../preprocessing/sample-data/sample-node.json"
EDGES = "../preprocessing/sample-data/sample-edge.csv"
SEED = 42

## Load users, edges and bot labels

In [ ]:
# Read IDs as strings: as numbers, 19-digit IDs get rounded.
predictions = pd.read_csv(PREDICTIONS, dtype={"user_id": str})
nodes = pd.read_json(NODES, dtype={"id": str})
edges = pd.read_csv(EDGES, dtype={"source_id": str, "target_id": str})

labelled_nodes = nodes.merge(predictions, left_on="id", right_on="user_id").rename(columns={"prediction": "label"})
print(f"{len(labelled_nodes)} of {len(nodes)} users labelled")
labelled_nodes["label"].value_counts().rename({0: "human", 1: "bot"})

## Keep reply edges between labelled users

The raw dataset stores reply targets as floats (e.g. `1.4835964632212644e+18`), so their last digits are lost.
We match them to users by comparing IDs as floats, then drop every edge that isn't a reply or doesn't
connect two users in our dataset.

In [ ]:
float_id_to_id = {float(uid): uid for uid in labelled_nodes["id"]}

replies = edges[edges["relation"] == "reply"].copy()
replies["source"] = replies["source_id"].astype(float).map(float_id_to_id)
replies["target"] = replies["target_id"].astype(float).map(float_id_to_id)
replies = replies.dropna(subset=["source", "target"])
print(f"{len(replies)} of {(edges['relation'] == 'reply').sum()} reply edges connect two labelled users")

In [ ]:
G = nx.Graph()
G.add_nodes_from((row.id, {"label": row.label}) for row in labelled_nodes.itertuples())
G.add_edges_from(zip(replies["source"], replies["target"]))
# Users with no reply edges can't belong to a community.
G.remove_nodes_from(list(nx.isolates(G)))
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## Communities with and without bots

In [ ]:
def louvain_summary(graph):
    partition = community_louvain.best_partition(graph, random_state=SEED)
    return len(set(partition.values())), community_louvain.modularity(partition, graph)

G_humans = G.copy()
G_humans.remove_nodes_from([n for n, d in G.nodes(data=True) if d["label"] == 1])
G_humans.remove_nodes_from(list(nx.isolates(G_humans)))

rows = []
for name, graph in [("humans + bots", G), ("humans only", G_humans)]:
    n_communities, modularity = louvain_summary(graph)
    rows.append({"graph": name, "nodes": graph.number_of_nodes(), "edges": graph.number_of_edges(),
                 "communities": n_communities, "modularity": round(modularity, 4)})
pd.DataFrame(rows)